In [1]:
%pip install meteostat pandas --quiet

Note: you may need to restart the kernel to use updated packages.


In [14]:
import requests
import pandas as pd

# 1. Configurações
lat = -20.8554
lon = -42.8005
start_date = "2017-05-01"
end_date = "2025-05-31"

url = "https://archive-api.open-meteo.com/v1/archive"

# 2. Seleção de Variáveis (Atmosfera + Solo)
daily_params = [
    # Atmosfera Básica
    "temperature_2m_mean", 
    "temperature_2m_max", 
    "temperature_2m_min", 
    "precipitation_sum", 
    
    # Energia e Demanda
    "et0_fao_evapotranspiration", 
    "shortwave_radiation_sum",     
    "vapor_pressure_deficit_max",
    
    # NOVO: Solo e Vento
    "soil_temperature_0_to_7cm_mean", # Temp na superfície do solo
    "soil_moisture_0_to_7cm_mean",    # Umidade superficial (m³/m³)
    "soil_moisture_7_to_28cm_mean",   # Umidade na zona principal da raiz (m³/m³)
    "wind_speed_10m_max"              # Rajada máxima de vento (importante para pulverização)
]

params = {
    "latitude": lat,
    "longitude": lon,
    "start_date": start_date,
    "end_date": end_date,
    "daily": daily_params,
    "timezone": "America/Sao_Paulo"
}

print("Baixando dataset completo (Atmosfera + Solo)...")
response = requests.get(url, params=params)

if response.status_code == 200:
    data = response.json()
    daily = data['daily']
    
    # Criando DataFrame
    df = pd.DataFrame({
        'Data': daily['time'],
        # Clima Aéreo
        'Temp_Media': daily['temperature_2m_mean'],
        'Temp_Max': daily['temperature_2m_max'],
        'Temp_Min': daily['temperature_2m_min'],
        'Chuva_mm': daily['precipitation_sum'],
        'ET0_mm': daily['et0_fao_evapotranspiration'],
        'Radiacao_MJ': daily['shortwave_radiation_sum'],
        'VPD_kPa': daily['vapor_pressure_deficit_max'],
        # Clima de Solo e Vento
        'Temp_Solo_C': daily['soil_temperature_0_to_7cm_mean'],
        'Umidade_Solo_Sup': daily['soil_moisture_0_to_7cm_mean'],
        'Umidade_Solo_Raiz': daily['soil_moisture_7_to_28cm_mean'],
        'Vento_Max_kmh': daily['wind_speed_10m_max']
    })
    
    df['Data'] = pd.to_datetime(df['Data'])
    
    # 3. Agregação Mensal
    regras = {
        'Temp_Media': 'mean',
        'Temp_Max': 'mean',
        'Temp_Min': 'mean',
        'Chuva_mm': 'sum',
        'ET0_mm': 'sum',
        'Radiacao_MJ': 'sum',
        'VPD_kPa': 'mean',          # Média do estresse
        'Temp_Solo_C': 'mean',      # Média térmica do solo
        'Umidade_Solo_Sup': 'mean', # Média da umidade volumétrica
        'Umidade_Solo_Raiz': 'mean',
        'Vento_Max_kmh': 'max'      # Pior rajada de vento do mês (ponto crítico)
    }
    
    df_mensal = df.resample('MS', on='Data').agg(regras).round(3) # 3 casas decimais para umidade do solo
    
    # Feature Engineering Extra: Eficiência de Chuva
    # Se choveu muito, mas a umidade do solo não subiu, houve escoamento (run-off)
    df_mensal['Indice_Retencao_Agua'] = df_mensal['Umidade_Solo_Raiz'] / (df_mensal['Chuva_mm'] + 1) # +1 evita div por zero
    
    df_mensal = df_mensal.reset_index()

    # Renomeação Profissional
    colunas_finais = {
        'Temp_Media': 'Temp_Ar_Media_C',
        'Umidade_Solo_Sup': 'Umidade_Solo_0_7cm_m3m3',
        'Umidade_Solo_Raiz': 'Umidade_Solo_7_28cm_m3m3',
        'Vento_Max_kmh': 'Vento_Rajada_Max_Mensal_kmh'
    }
    df_mensal = df_mensal.rename(columns=colunas_finais)

    print(f"\nSucesso! Dataset 'Clima' gerado com {len(df_mensal)} linhas.")
    print(df_mensal[['Data', 'Chuva_mm', 'Umidade_Solo_7_28cm_m3m3', 'Vento_Rajada_Max_Mensal_kmh']].head())

    df_mensal.to_csv('clima_coimbra.csv', index=False, sep=';', decimal='.', encoding='latin-1')
    print("Arquivo salvo: clima_coimbra.csv")

else:
    print(f"Erro: {response.status_code}")

Baixando dataset completo (Atmosfera + Solo)...

Sucesso! Dataset 'Clima' gerado com 97 linhas.
        Data  Chuva_mm  Umidade_Solo_7_28cm_m3m3  Vento_Rajada_Max_Mensal_kmh
0 2017-05-01      37.5                     0.349                         25.3
1 2017-06-01      17.3                     0.336                         26.3
2 2017-07-01       7.1                     0.329                         20.6
3 2017-08-01       2.4                     0.313                         35.0
4 2017-09-01      16.1                     0.289                         27.1
Arquivo salvo: clima_coimbra.csv


In [15]:
# 1. Definição da Lógica de Safra
# Regra: Se o mês é >= Maio (5), pertence à safra do ano seguinte.
# Ex: Maio/2023 inicia o ciclo da Safra 2024.
# Ex: Abril/2024 encerra o ciclo da Safra 2024.

def definir_safra(row):
    # Se estamos em Maio ou depois, já estamos "trabalhando" para a safra do ano seguinte
    if row['Data'].month >= 5:
        return row['Data'].year + 1
    else:
        return row['Data'].year

# Aplicando a lógica ao DataFrame diário (df) gerado anteriormente
df['Ano_Safra'] = df.apply(definir_safra, axis=1)

# 2. Feature Engineering Avançada (Variáveis Derivadas)
# Contagem de dias com risco de geada (Tmin < 2°C) - Crítico para Coimbra
df['Dia_Geada'] = df['Temp_Min'] < 2.0
# Contagem de dias com estresse térmico (Tmax > 32°C) - Abortamento de florada
df['Dia_Calor_Extremo'] = df['Temp_Max'] > 32.0

# 3. Agregação por Safra
# Aqui a mágica acontece: transformamos 365 linhas em 1 linha de resumo da safra
regras_safra = {
    'Chuva_mm': 'sum',                # Chuva Acumulada na safra
    'ET0_mm': 'sum',                  # Demanda Hídrica Total
    'Temp_Media': 'mean',             # Temperatura média do ciclo
    'Temp_Max': 'max',                # A maior temperatura absoluta registrada na safra
    'Temp_Min': 'min',                # A menor temperatura absoluta (risco de geada)
    'Dia_Geada': 'sum',               # Total de dias com risco de geada
    'Dia_Calor_Extremo': 'sum',       # Total de dias de calor excessivo
    'Radiacao_MJ': 'sum',             # Energia solar total acumulada
    'VPD_kPa': 'mean',                # Estresse médio
    'Umidade_Solo_Raiz': 'mean',      # Disponibilidade média de água no solo
    'Vento_Max_kmh': 'max'            # A rajada mais forte do ano
}

df_safra = df.groupby('Ano_Safra').agg(regras_safra).round(2)

# 4. Cálculos Pós-Agregação (Balanço Hídrico Anual)
# P - ET0: O saldo final de água da safra
df_safra['Deficit_Hidrico_Anual_mm'] = df_safra['Chuva_mm'] - df_safra['ET0_mm']

# Resetando o index para ter "Ano_Safra" como coluna
df_safra = df_safra.reset_index()

# 5. Limpeza de Safras Incompletas
# Como seus dados começam em Maio/2018 e vão até Maio/2025:
# Safra 2019 (Maio/18 a Abril/19): Completa.
# Safra 2025 (Maio/24 a Abril/25): Completa.
# Safra 2026 (Começa em Maio/25): Incompleta (só tem 1 mês). Devemos remover.

# Contamos quantos dias tem em cada safra para verificar integridade
contagem_dias = df.groupby('Ano_Safra').size()
# Filtramos safras que tenham pelo menos 360 dias (para garantir ano completo)
safras_validas = contagem_dias[contagem_dias >= 360].index
df_safra = df_safra[df_safra['Ano_Safra'].isin(safras_validas)]

# 6. Renomeação para o Modelo de Clusterização
mapa_nomes = {
    'Chuva_mm': 'Precipitacao_Acumulada_mm',
    'Temp_Media': 'Temp_Media_Ciclo_C',
    'Temp_Max': 'Temp_Absoluta_Max_C',
    'Temp_Min': 'Temp_Absoluta_Min_C',
    'Dia_Geada': 'Dias_Risco_Geada',
    'Dia_Calor_Extremo': 'Dias_Calor_Extremo',
    'Umidade_Solo_Raiz': 'Umidade_Solo_Media_m3m3'
}
df_safra = df_safra.rename(columns=mapa_nomes)

print(f"Sucesso! Base consolidada por Safra gerada.")
print(df_safra.head(10))

# Exportação Final
df_safra.to_csv('dados_safra_cafe_coimbra.csv', index=False, sep=';', decimal='.', encoding='latin-1')

Sucesso! Base consolidada por Safra gerada.
   Ano_Safra  Precipitacao_Acumulada_mm   ET0_mm  Temp_Media_Ciclo_C  \
0       2018                     1063.4  1306.88               20.32   
1       2019                     1097.8  1328.54               20.74   
2       2020                     1365.8  1308.46               20.60   
3       2021                     1183.1  1386.51               20.42   
4       2022                     1455.9  1348.52               20.05   
5       2023                     1401.1  1353.02               19.88   
6       2024                     1493.5  1398.15               20.90   
7       2025                      947.1  1452.10               20.96   

   Temp_Absoluta_Max_C  Temp_Absoluta_Min_C  Dias_Risco_Geada  \
0                 32.5                  8.6                 0   
1                 32.5                  9.0                 0   
2                 34.4                  6.6                 0   
3                 34.8                  8.0    